In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

### Create RDD from elements in memory

In [4]:
rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 4, 4, 2], 4)

myCollection = "Spark Unified Computation Engine : Big Data Processing Made Simple".split(" ")
words = spark.sparkContext.parallelize(myCollection, 2)

### Filter Transformation using custom filter function

In [5]:
def startsWithS(element):
  return element.startswith("S")

In [6]:
filteredWords = words.filter(lambda x: startsWithS(x))
filteredWords.collect()

['Spark', 'Simple']

### Map Transformation

In [7]:
mapTransformation = words.map(lambda x: (x, x[0], startsWithS(x)))
mapTransformation.collect()

[('Spark', 'S', True),
 ('Unified', 'U', False),
 ('Computation', 'C', False),
 ('Engine', 'E', False),
 (':', ':', False),
 ('Big', 'B', False),
 ('Data', 'D', False),
 ('Processing', 'P', False),
 ('Made', 'M', False),
 ('Simple', 'S', True)]

### Take Action

In [8]:
mapTransformation.take(5)

[('Spark', 'S', True),
 ('Unified', 'U', False),
 ('Computation', 'C', False),
 ('Engine', 'E', False),
 (':', ':', False)]

### Perform sorting

- Ascending

In [9]:
words.sortBy(lambda x: len(x)).collect()

[':',
 'Big',
 'Data',
 'Made',
 'Spark',
 'Engine',
 'Simple',
 'Unified',
 'Processing',
 'Computation']

- Descending

In [10]:
words.sortBy(lambda x: len(x) * -1).collect()

['Computation',
 'Processing',
 'Unified',
 'Engine',
 'Simple',
 'Spark',
 'Data',
 'Made',
 'Big',
 ':']

### Perform Reduce Action

In [11]:
# Sum of all numbers from 1 to 20
spark.sparkContext.parallelize(range(1, 21)).reduce(lambda x, y: x + y)

210

In [12]:
# To get the word with highest length
words.reduce(lambda x, y: x if len(x) > len(y) else y)

'Computation'

### mapPartitions
- To apply a function to each partition of the RDD
- It passes the elemets of the partition as an iterator to the function
- Helpful in scenarios like connecting to a database before starting some processing on a transformation and closing the DB connection once transformation completed

In [ ]:
def processingFunction(iter):
  yield "Connecting to the SQL DB..."
  for i in iter:
    yield (len(i), i)
  yield "Closing the SQL DB connection..."

words.mapPartitions(processingFunction).collect()
# As shown below the SQL DB connection is created and closed for each partition of an RDD

['Connecting to the SQL DB...',
 (5, 'Spark'),
 (7, 'Unified'),
 (11, 'Computation'),
 (6, 'Engine'),
 (1, ':'),
 'Closing the SQL DB connection...',
 'Connecting to the SQL DB...',
 (3, 'Big'),
 (4, 'Data'),
 (10, 'Processing'),
 (4, 'Made'),
 (6, 'Simple'),
 'Closing the SQL DB connection...']

### mapPartitionsWithIndex
- This is similar to mappartitions
- Along with elemet iterator it passes the partition index to the transformation function
- This helps to apply custom logic based on the index of the partition

In [ ]:
def processingFunction(idx, iter):
  yield "Connecting to the SQL DB..."
  if idx == 0:
    yield "This is custom logic only for the 1st partition"
  else:
    yield "This is custom logic for rest of the partitions"
  for i in iter:
    yield (idx, len(i), i)
  yield "Closing the SQL DB connection..."

words.mapPartitionsWithIndex(processingFunction).collect()


['Connecting to the SQL DB...',
 'This is custom logic only for the 1st partition',
 (0, 5, 'Spark'),
 (0, 7, 'Unified'),
 (0, 11, 'Computation'),
 (0, 6, 'Engine'),
 (0, 1, ':'),
 'Closing the SQL DB connection...',
 'Connecting to the SQL DB...',
 'This is custom logic for rest of the partitions',
 (1, 3, 'Big'),
 (1, 4, 'Data'),
 (1, 10, 'Processing'),
 (1, 4, 'Made'),
 (1, 6, 'Simple'),
 'Closing the SQL DB connection...']

### Additional RDD actions

In [15]:
rdd = spark.sparkContext.parallelize([100, 2, 3, 3, 410, 3, 3, 3, 4, 104, 2])

- Count the no. of elements in the RDD

In [16]:
rdd.count()

11

- Get distinct values

In [17]:
distinct_rdd = rdd.distinct() # Transformation
distinct_rdd.collect() # Action

[100, 2, 410, 4, 104, 3]

- Count of distinct elements

In [18]:
rdd.distinct().count()

6

- Get first element

In [19]:
rdd.first()

100

- Limit the output

In [20]:
rdd.take(5)

[100, 2, 3, 3, 410]

- Get count of each element

In [21]:
rdd.countByValue()

defaultdict(int, {100: 1, 2: 2, 3: 5, 410: 1, 4: 1, 104: 1})

- Get MIN, MAX, MEAN & SUM of the RDD elements

In [22]:
print(f'Min: {rdd.min()}, Max: {rdd.max()},  Mean: {rdd.mean()}, Sum: {rdd.sum()}')

Min: 2, Max: 410,  Mean: 57.90909090909091, Sum: 637


- Sort the RDD in ascending order and take fisrt N elements

In [23]:
rdd.takeOrdered(5)

[2, 2, 3, 3, 3]

- Sort the RDD in descending order and take fisrt N elements (Similar to top())

In [24]:
rdd.takeOrdered(5, lambda x : -x)

[410, 104, 100, 4, 3]

- Get top N values from the RDD: It returns list in descending order

In [ ]:
rdd.top(5)

[410, 104, 100, 4, 3]